# MBUSA Case Study
### Coded by Derick Stevens
---
This notebook serves as a showing of capability and skill regarding exploratory data analysis on a sample set of data from MBUSA. The goal is to find anomalies in the dataset and briefly explain how those anomalies could have came about.

### Initial Imports

In [1]:
import polars as pl
import plotly.express as px
from ipywidgets import widgets

pl.Config.set_tbl_rows(-1)

polars.config.Config

### Data Collection
This section is to set up the initial tables and get a clean-joined view of the data.

In [2]:
case_dict = pl.read_excel(
    "Pricing Analytics Business Case data vfinal.xlsx",
    sheet_name=['Dealers', 'Models', 'Inventory', 'Sales', 'Interest_Rates', 'Forecast_Assumptions']
)

# I want to split these out for individual EDA, but I will still group them to get different views
dealers = case_dict['Dealers']
models = case_dict['Models']
inventory = case_dict['Inventory']
sales = case_dict['Sales']
interest_rates = case_dict['Interest_Rates']
forecast_assumptions = case_dict['Forecast_Assumptions']

# showing the first ten sales by region, dealer name and date sold.
sales_with_region = sales.join(dealers, on='dealer_id', how='inner')
sales_with_region  = sales_with_region.join(models, on = 'model_code', how = 'inner')
sales_with_region = sales_with_region.with_columns(
    pl.col('sale_date')
    .str.to_date()
    .dt.strftime('%Y-%m')
    .alias('month')
)
print('Showing the first ten sales with region included, we are going to do some analysis vs the forecasted amount for that month.\n')
print(sales_with_region.head(10))
print(sales_with_region.select(pl.col('sale_price')).describe())

Showing the first ten sales with region included, we are going to do some analysis vs the forecasted amount for that month.

shape: (10, 9)
┌─────────┬────────────┬───────────┬────────────┬───┬─────────────┬─────────┬────────────┬─────────┐
│ sale_id ┆ sale_date  ┆ dealer_id ┆ model_code ┆ … ┆ dealer_name ┆ region  ┆ model_name ┆ month   │
│ ---     ┆ ---        ┆ ---       ┆ ---        ┆   ┆ ---         ┆ ---     ┆ ---        ┆ ---     │
│ i64     ┆ str        ┆ i64       ┆ str        ┆   ┆ str         ┆ str     ┆ str        ┆ str     │
╞═════════╪════════════╪═══════════╪════════════╪═══╪═════════════╪═════════╪════════════╪═════════╡
│ 600000  ┆ 2024-05-08 ┆ 308       ┆ MODEL_9    ┆ … ┆ MB Dealer   ┆ East    ┆ Model 9    ┆ 2024-05 │
│         ┆            ┆           ┆            ┆   ┆ 308         ┆         ┆            ┆         │
│ 600001  ┆ 2025-12-17 ┆ 302       ┆ MODEL_9    ┆ … ┆ MB Dealer   ┆ Midwest ┆ Model 9    ┆ 2025-12 │
│         ┆            ┆           ┆            ┆   

### Light Exploration
I want to start off by looking through different combos of data to see if I can easily find any outliers. The first set is just a general bird's eye view look at the data.

In [3]:
regional_sale_volume = sales_with_region.group_by(['region']).agg(
    pl.col("model_code").count().alias("regional_sales"),
    pl.col('sale_price').mean().round().alias("avg_sale_price")
).sort(by=['avg_sale_price'], descending=True)
print("Region Sales w/ avg Sale Price.\n\n", regional_sale_volume, "\n\n")

Region Sales w/ avg Sale Price.

 shape: (4, 3)
┌─────────┬────────────────┬────────────────┐
│ region  ┆ regional_sales ┆ avg_sale_price │
│ ---     ┆ ---            ┆ ---            │
│ str     ┆ u32            ┆ f64            │
╞═════════╪════════════════╪════════════════╡
│ West    ┆ 1277           ┆ 45059.0        │
│ East    ┆ 1262           ┆ 44901.0        │
│ South   ┆ 1225           ┆ 44882.0        │
│ Midwest ┆ 1236           ┆ 44878.0        │
└─────────┴────────────────┴────────────────┘ 




In [4]:
px.bar(
    regional_sale_volume,
    x='region',
    y='regional_sales',
    color='region',
)

In [5]:
print("Or average sale price...")
px.bar(
    regional_sale_volume,
    x='region',
    y='avg_sale_price',
    color='region',
)

Or average sale price...


I can see there isn't much variation in the overall data. Let's dig a little deeper and split-out the model code.

In [6]:
actual_sale_count_by_region = sales_with_region.group_by(['region', 'model_code']).agg(
    pl.col("model_code").count().alias("regional_sales")
).sort(by=['model_code', 'region'], descending=False)
print("Model Sales by Region\n\n", actual_sale_count_by_region, "\n\n")

Model Sales by Region

 shape: (40, 3)
┌─────────┬────────────┬────────────────┐
│ region  ┆ model_code ┆ regional_sales │
│ ---     ┆ ---        ┆ ---            │
│ str     ┆ str        ┆ u32            │
╞═════════╪════════════╪════════════════╡
│ East    ┆ MODEL_1    ┆ 125            │
│ Midwest ┆ MODEL_1    ┆ 130            │
│ South   ┆ MODEL_1    ┆ 116            │
│ West    ┆ MODEL_1    ┆ 130            │
│ East    ┆ MODEL_10   ┆ 136            │
│ Midwest ┆ MODEL_10   ┆ 111            │
│ South   ┆ MODEL_10   ┆ 108            │
│ West    ┆ MODEL_10   ┆ 146            │
│ East    ┆ MODEL_2    ┆ 120            │
│ Midwest ┆ MODEL_2    ┆ 96             │
│ South   ┆ MODEL_2    ┆ 139            │
│ West    ┆ MODEL_2    ┆ 116            │
│ East    ┆ MODEL_3    ┆ 131            │
│ Midwest ┆ MODEL_3    ┆ 130            │
│ South   ┆ MODEL_3    ┆ 116            │
│ West    ┆ MODEL_3    ┆ 121            │
│ East    ┆ MODEL_4    ┆ 120            │
│ Midwest ┆ MODEL_4    ┆ 130         

In [7]:
px.scatter(
    actual_sale_count_by_region,
    x='model_code',
    y='regional_sales',
    color='region',
)

### What Do I See?

Visually, I am looking at groups with wide variations and values that exceed 140 units or fall below 100 units. There aren't many that stick out, but I will single out:
- MIDWEST - MODEL_2 // *LOW*
- EAST - MODEL_6 // *LOW*
- WEST - MODEL_8 // *LOW*
- WEST - MODEL_10 // *HIGH*
- MIDWEST - MODEL_5 // *HIGH*
- WEST - MODEL_6 // *HIGH*
- EAST - MODEL_8 // *HIGH*
- WEST - MODEL_9 // *HIGH*

More views are needed to see why this volume could be so low in comparison. High sales volume should be considered too, what if those values have high volume but a low average selling price that doesn't meet expectation? Without doing an STD or looking at the AVG sales volume (in total), I am just picking out points that look a bit high and a bit low. At this point, I am not quite sure what the cause for this is, but it gives me some indication of what to keep my eye on when looking at the set.

In [8]:
actual_sale_count_by_dealer = sales_with_region.group_by(['region', 'model_code', 'dealer_id', 'dealer_name']).agg(
    pl.col("model_code").count().alias("dealer_sales")
).sort(by=['model_code', 'region', 'dealer_id'], descending=False)
print("Looking at dealer sales volume over the year to see if anything sticks out...\n\n", actual_sale_count_by_dealer, "\n\n")

actual_sale_count_by_dealer = actual_sale_count_by_dealer.with_columns([
    (pl.col("dealer_sales").mean().alias("dealer_sale_avg")),
])
actual_sale_count_by_dealer = actual_sale_count_by_dealer.with_columns(
    (pl.when(
        pl.col("dealer_sales") < pl.col("dealer_sale_avg")
    ).then(pl.lit("UNDER")).when(pl.col("dealer_sales") >= pl.col("dealer_sale_avg")).then(pl.lit("OVER"))
    ).alias("avg_check")
)

Looking at dealer sales volume over the year to see if anything sticks out...

 shape: (200, 5)
┌─────────┬────────────┬───────────┬───────────────┬──────────────┐
│ region  ┆ model_code ┆ dealer_id ┆ dealer_name   ┆ dealer_sales │
│ ---     ┆ ---        ┆ ---       ┆ ---           ┆ ---          │
│ str     ┆ str        ┆ i64       ┆ str           ┆ u32          │
╞═════════╪════════════╪═══════════╪═══════════════╪══════════════╡
│ East    ┆ MODEL_1    ┆ 300       ┆ MB Dealer 300 ┆ 25           │
│ East    ┆ MODEL_1    ┆ 304       ┆ MB Dealer 304 ┆ 30           │
│ East    ┆ MODEL_1    ┆ 308       ┆ MB Dealer 308 ┆ 21           │
│ East    ┆ MODEL_1    ┆ 312       ┆ MB Dealer 312 ┆ 29           │
│ East    ┆ MODEL_1    ┆ 316       ┆ MB Dealer 316 ┆ 20           │
│ Midwest ┆ MODEL_1    ┆ 302       ┆ MB Dealer 302 ┆ 25           │
│ Midwest ┆ MODEL_1    ┆ 306       ┆ MB Dealer 306 ┆ 28           │
│ Midwest ┆ MODEL_1    ┆ 310       ┆ MB Dealer 310 ┆ 28           │
│ Midwest ┆ MODEL_1 

In [9]:
px.scatter(
    actual_sale_count_by_dealer,
    x='model_code',
    y='dealer_sales',
    color='avg_check',
    hover_name='dealer_id',
    hover_data='dealer_name'
)

In [10]:
print("Total Dealer Sales over AVG", actual_sale_count_by_dealer.select(pl.col("avg_check")).filter(pl.col("avg_check") == "OVER").count(), '\n',
"Total Dealer Sales under AVG", actual_sale_count_by_dealer.select(pl.col("avg_check")).filter(pl.col("avg_check") == "UNDER").count())

Total Dealer Sales over AVG shape: (1, 1)
┌───────────┐
│ avg_check │
│ ---       │
│ u32       │
╞═══════════╡
│ 103       │
└───────────┘ 
 Total Dealer Sales under AVG shape: (1, 1)
┌───────────┐
│ avg_check │
│ ---       │
│ u32       │
╞═══════════╡
│ 97        │
└───────────┘


The actual dealer sales volume seems to be symmetrical. My assumption here is that outliers would land in a z-score of +2. Let's see if there is another way to get to our outliers.

# Starting to Look at the Forecast

## Velocity, velocity, velocity.

How fast are we selling and what models are the hottest selling?

In [11]:
sale_velocity = actual_sale_count_by_dealer.with_columns(
    (pl.col("dealer_sales")/12).round(decimals=2).alias("avg_sales_per_month")
)
sale_velocity = sale_velocity.join(inventory, on=['dealer_id', 'model_code'])
sale_velocity = sale_velocity.with_columns(
    (pl.col('on_hand')/pl.col('avg_sales_per_month')).round().alias("months_to_exhaust_inventory")
)
sale_velocity = sale_velocity.drop(['dealer_sale_avg', 'avg_check'])
print(sale_velocity)

'''
dataframe for seeing % of units sold, which models are selling the fastest (which can help optimize the forecasting model), which region has the highest avg sell rate, which model has the highest avg sell rate over time, forecast assumed burn to see how close the sell rate is between the model and dealers
- forecasted sell rate
- actual sell rate
per model...
'''

shape: (200, 8)
┌─────────┬────────────┬───────────┬─────────────┬─────────────┬────────────┬─────────┬────────────┐
│ region  ┆ model_code ┆ dealer_id ┆ dealer_name ┆ dealer_sale ┆ avg_sales_ ┆ on_hand ┆ months_to_ │
│ ---     ┆ ---        ┆ ---       ┆ ---         ┆ s           ┆ per_month  ┆ ---     ┆ exhaust_in │
│ str     ┆ str        ┆ i64       ┆ str         ┆ ---         ┆ ---        ┆ i64     ┆ ventory    │
│         ┆            ┆           ┆             ┆ u32         ┆ f64        ┆         ┆ ---        │
│         ┆            ┆           ┆             ┆             ┆            ┆         ┆ f64        │
╞═════════╪════════════╪═══════════╪═════════════╪═════════════╪════════════╪═════════╪════════════╡
│ East    ┆ MODEL_1    ┆ 300       ┆ MB Dealer   ┆ 25          ┆ 2.08       ┆ 8       ┆ 4.0        │
│         ┆            ┆           ┆ 300         ┆             ┆            ┆         ┆            │
│ East    ┆ MODEL_2    ┆ 300       ┆ MB Dealer   ┆ 21          ┆ 1.75      

'\ndataframe for seeing % of units sold, which models are selling the fastest (which can help optimize the forecasting model), which region has the highest avg sell rate, which model has the highest avg sell rate over time, forecast assumed burn to see how close the sell rate is between the model and dealers\n- forecasted sell rate\n- actual sell rate\nper model...\n'

In [12]:
forecasted_sale_velo = forecast_assumptions.drop(['month', 'avg_price_assumption', 'apr_assumption'])
forecasted_sale_velo = forecasted_sale_velo.group_by(['region', 'model_code']).agg(
    pl.col("units_forecast").sum().alias("total_units_forecast")
)
forecasted_sale_velo = forecasted_sale_velo.with_columns(
    (pl.col('total_units_forecast')/12).round(decimals=2).alias("forecasted_sales_per_month")
)
print(forecasted_sale_velo.head())

shape: (5, 4)
┌────────┬────────────┬──────────────────────┬────────────────────────────┐
│ region ┆ model_code ┆ total_units_forecast ┆ forecasted_sales_per_month │
│ ---    ┆ ---        ┆ ---                  ┆ ---                        │
│ str    ┆ str        ┆ i64                  ┆ f64                        │
╞════════╪════════════╪══════════════════════╪════════════════════════════╡
│ South  ┆ MODEL_2    ┆ 491                  ┆ 40.92                      │
│ West   ┆ MODEL_8    ┆ 577                  ┆ 48.08                      │
│ West   ┆ MODEL_9    ┆ 595                  ┆ 49.58                      │
│ East   ┆ MODEL_8    ┆ 486                  ┆ 40.5                       │
│ South  ┆ MODEL_4    ┆ 527                  ┆ 43.92                      │
└────────┴────────────┴──────────────────────┴────────────────────────────┘


In [13]:
sale_velocity = sale_velocity.join(forecasted_sale_velo, on=['region', 'model_code'])
print(sale_velocity.head(65))

shape: (65, 10)
┌─────────┬────────────┬───────────┬────────────┬───┬─────────┬────────────┬───────────┬───────────┐
│ region  ┆ model_code ┆ dealer_id ┆ dealer_nam ┆ … ┆ on_hand ┆ months_to_ ┆ total_uni ┆ forecaste │
│ ---     ┆ ---        ┆ ---       ┆ e          ┆   ┆ ---     ┆ exhaust_in ┆ ts_foreca ┆ d_sales_p │
│ str     ┆ str        ┆ i64       ┆ ---        ┆   ┆ i64     ┆ ventory    ┆ st        ┆ er_month  │
│         ┆            ┆           ┆ str        ┆   ┆         ┆ ---        ┆ ---       ┆ ---       │
│         ┆            ┆           ┆            ┆   ┆         ┆ f64        ┆ i64       ┆ f64       │
╞═════════╪════════════╪═══════════╪════════════╪═══╪═════════╪════════════╪═══════════╪═══════════╡
│ East    ┆ MODEL_1    ┆ 300       ┆ MB Dealer  ┆ … ┆ 8       ┆ 4.0        ┆ 483       ┆ 40.25     │
│         ┆            ┆           ┆ 300        ┆   ┆         ┆            ┆           ┆           │
│ East    ┆ MODEL_2    ┆ 300       ┆ MB Dealer  ┆ … ┆ 24      ┆ 14.0       

In [14]:
actual_regional_sale_count_by_month_with_forecast = sales_with_region.group_by(['region', 'model_code', 'month']).agg(
    pl.col("model_code").count().alias("units_actual"),
    pl.col("sale_price").mean().round().alias("avg_sale_price"),
).sort(by=['model_code', 'region', 'month'], descending=False)

forecast_with_actual_sc = actual_regional_sale_count_by_month_with_forecast.join(forecast_assumptions, on=['region', 'model_code', 'month'], how='inner')

forecast_with_actual_sc = forecast_with_actual_sc.with_columns([
    ((pl.col('units_actual') - pl.col('units_forecast')).alias('unit_difference')),
    ((pl.col('avg_sale_price') - pl.col('avg_price_assumption')).alias('avg_price_difference'))
])

print(forecast_with_actual_sc.head())


shape: (5, 10)
┌────────┬────────────┬─────────┬────────────┬───┬────────────┬────────────┬───────────┬───────────┐
│ region ┆ model_code ┆ month   ┆ units_actu ┆ … ┆ avg_price_ ┆ apr_assump ┆ unit_diff ┆ avg_price │
│ ---    ┆ ---        ┆ ---     ┆ al         ┆   ┆ assumption ┆ tion       ┆ erence    ┆ _differen │
│ str    ┆ str        ┆ str     ┆ ---        ┆   ┆ ---        ┆ ---        ┆ ---       ┆ ce        │
│        ┆            ┆         ┆ u32        ┆   ┆ f64        ┆ f64        ┆ i64       ┆ ---       │
│        ┆            ┆         ┆            ┆   ┆            ┆            ┆           ┆ f64       │
╞════════╪════════════╪═════════╪════════════╪═══╪════════════╪════════════╪═══════════╪═══════════╡
│ East   ┆ MODEL_1    ┆ 2025-01 ┆ 6          ┆ … ┆ 40984.07   ┆ 4.556      ┆ -13       ┆ 5407.93   │
│ East   ┆ MODEL_1    ┆ 2025-02 ┆ 6          ┆ … ┆ 40820.85   ┆ 6.033      ┆ -59       ┆ 3459.15   │
│ East   ┆ MODEL_1    ┆ 2025-03 ┆ 3          ┆ … ┆ 55018.6    ┆ 4.174      ┆

## How about We Use a Bucket?

I want to see which sales could be considered optimal. This exercise is going off of the assumption of a few things and following these stipulations:
- A high price and a high APR is a good thing in combination (for us)
- A low price and a high APR is neutral
- A low price and a low APR is a good day for the customer
- A high price and a low APR is neutral
- A sale price is considered high when it exceeds the assumed price from the model and vice versa.
- There will be 3 buckets. Optimal, Neutral and Suboptimal
- The sales data does not tell me the terms of the deal so I will be using the avg APR in that region (published) as a proxy.
- Lastly, I am assuming that all values are Financed amounts. This *will* skew the data as I'm sure some leases are there.

In [15]:
interest_rates_cleaned = interest_rates.filter(pl.col("product_type") != "Lease")
interest_rates_cleaned = interest_rates_cleaned.group_by(['month', 'region', 'model_code']).agg(
    pl.col("published_apr").mean().round(decimals=2).alias("proxy_apr"),
).sort(by=['month','model_code','region'])

bucketing = interest_rates_cleaned.join(sales_with_region, on=['month', 'region', 'model_code'], how='inner').sort(by=['month', 'region', 'model_code'], descending=False)
bucketing = forecast_assumptions.join(bucketing, on=['month', 'region', 'model_code'], how='inner')
bucketing.drop('units_forecast')
bucketing = bucketing.with_columns([
    (pl.col('sale_price') - pl.col('avg_price_assumption')).alias('sale_price_difference'),
    (pl.col('proxy_apr') - pl.col('apr_assumption')).alias('apr_difference'),
])
print(bucketing.head(25))

shape: (25, 15)
┌─────────┬────────┬────────────┬────────────┬───┬────────────┬────────────┬───────────┬───────────┐
│ month   ┆ region ┆ model_code ┆ units_fore ┆ … ┆ dealer_nam ┆ model_name ┆ sale_pric ┆ apr_diffe │
│ ---     ┆ ---    ┆ ---        ┆ cast       ┆   ┆ e          ┆ ---        ┆ e_differe ┆ rence     │
│ str     ┆ str    ┆ str        ┆ ---        ┆   ┆ ---        ┆ str        ┆ nce       ┆ ---       │
│         ┆        ┆            ┆ i64        ┆   ┆ str        ┆            ┆ ---       ┆ f64       │
│         ┆        ┆            ┆            ┆   ┆            ┆            ┆ f64       ┆           │
╞═════════╪════════╪════════════╪════════════╪═══╪════════════╪════════════╪═══════════╪═══════════╡
│ 2025-01 ┆ East   ┆ MODEL_1    ┆ 19         ┆ … ┆ MB Dealer  ┆ Model 1    ┆ 10253.17  ┆ 2.214     │
│         ┆        ┆            ┆            ┆   ┆ 304        ┆            ┆           ┆           │
│ 2025-01 ┆ East   ┆ MODEL_1    ┆ 19         ┆ … ┆ MB Dealer  ┆ Model 1    

In [16]:
bucketing = bucketing.with_columns(
    pl.when((pl.col('sale_price_difference') > 0) & (pl.col("proxy_apr") > pl.col("apr_assumption")))
    .then(pl.lit('Optimal'))
    .when((pl.col('sale_price_difference') < 0) & (pl.col("proxy_apr") < pl.col("apr_assumption")))
    .then(pl.lit('Suboptimal'))
    .otherwise(pl.lit('Neutral')).alias('sale_bucket')
)

print(bucketing.head(25))

shape: (25, 16)
┌─────────┬────────┬────────────┬────────────┬───┬────────────┬────────────┬───────────┬───────────┐
│ month   ┆ region ┆ model_code ┆ units_fore ┆ … ┆ model_name ┆ sale_price ┆ apr_diffe ┆ sale_buck │
│ ---     ┆ ---    ┆ ---        ┆ cast       ┆   ┆ ---        ┆ _differenc ┆ rence     ┆ et        │
│ str     ┆ str    ┆ str        ┆ ---        ┆   ┆ str        ┆ e          ┆ ---       ┆ ---       │
│         ┆        ┆            ┆ i64        ┆   ┆            ┆ ---        ┆ f64       ┆ str       │
│         ┆        ┆            ┆            ┆   ┆            ┆ f64        ┆           ┆           │
╞═════════╪════════╪════════════╪════════════╪═══╪════════════╪════════════╪═══════════╪═══════════╡
│ 2025-01 ┆ East   ┆ MODEL_1    ┆ 19         ┆ … ┆ Model 1    ┆ 10253.17   ┆ 2.214     ┆ Optimal   │
│ 2025-01 ┆ East   ┆ MODEL_1    ┆ 19         ┆ … ┆ Model 1    ┆ 1429.25    ┆ 2.214     ┆ Optimal   │
│ 2025-01 ┆ East   ┆ MODEL_1    ┆ 19         ┆ … ┆ Model 1    ┆ -1657.24   

In [17]:
colz = {'Optimal':'green', 'Neutral':'blue', 'Suboptimal':'red'}

px.scatter(
    bucketing,
    x='sale_price_difference',
    y='apr_difference',
    color='sale_bucket',
    color_discrete_map=colz,
    hover_name='sale_id',
    hover_data=['model_code', 'region', 'sale_price'],

)

In [18]:
buckcounts = bucketing.group_by("sale_bucket").agg(
    pl.col('sale_bucket').count().alias("bucket_count"),
)
buckcounts = buckcounts.with_columns(
    ((pl.col("bucket_count")/pl.col('bucket_count').sum()) * 100).round().alias("percent_bucket_count")
)
print(buckcounts)

shape: (3, 3)
┌─────────────┬──────────────┬──────────────────────┐
│ sale_bucket ┆ bucket_count ┆ percent_bucket_count │
│ ---         ┆ ---          ┆ ---                  │
│ str         ┆ u32          ┆ f64                  │
╞═════════════╪══════════════╪══════════════════════╡
│ Neutral     ┆ 1239         ┆ 50.0                 │
│ Suboptimal  ┆ 425          ┆ 17.0                 │
│ Optimal     ┆ 801          ┆ 32.0                 │
└─────────────┴──────────────┴──────────────────────┘


### Questions:
- For low values, those that are sub lower fence, are those Leases, CPO, Trade-Ins, partial cash deals?
- Shouldn't there be a consideration for leases in the forecast model, especially considering there is an avg being used?
- What is the optimal % for vehicles on lot versus vehicles sold?